# 实验追踪教程

> **前置知识**: Python基础、机器学习训练流程
>
> **学习目标**: 掌握实验追踪的核心概念和实现方法

---

## 为什么需要实验追踪？

```
传统ML开发的痛点:
┌─────────────────────────────────────────────────────────────┐
│  "上周那个效果好的模型用的什么参数？" → 忘了              │
│  "这次训练为什么比上次差？"           → 没法比较          │
│  "能复现三个月前的实验吗？"           → 不能              │
└─────────────────────────────────────────────────────────────┘

实验追踪解决方案:
┌─────────────────────────────────────────────────────────────┐
│  记录每次实验的:                                             │
│  ├── 超参数配置 (learning_rate, batch_size, ...)           │
│  ├── 训练指标 (loss, accuracy, ...)                        │
│  ├── 模型文件 (model.pt, checkpoint.pth, ...)              │
│  └── 代码/数据版本                                          │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **Experiment 数据类** - 理解实验记录的数据结构
2. **ExperimentTracker** - 本地文件系统追踪器
3. **参数和指标记录** - log_params, log_metrics
4. **实验比较** - 对比多个实验结果
5. **MLflow 集成** - 企业级追踪方案

In [ ]:
# ============================================================
# 环境准备
# ============================================================
# 添加源码路径，使得可以导入 src 目录下的模块
import sys
import os
sys.path.insert(0, '../src')

# 标准库
import time
import tempfile  # 创建临时目录，用于演示
import shutil    # 文件操作，用于清理
from pathlib import Path

# 导入实验追踪模块
from experiment_tracker import (
    Experiment,           # 实验数据类：存储实验的所有信息
    ExperimentStatus,     # 实验状态枚举：RUNNING, COMPLETED, FAILED
    ExperimentTracker,    # 本地追踪器：基于文件系统
    ExperimentComparator, # 实验比较器：对比多个实验
    create_tracker,       # 工厂函数：创建追踪器
    MLFLOW_AVAILABLE,     # MLflow 是否可用
    WANDB_AVAILABLE,      # Weights & Biases 是否可用
)

# 检查可选依赖
print("=" * 50)
print("环境检查")
print("=" * 50)
print(f"MLflow 可用: {MLFLOW_AVAILABLE}")
print(f"W&B 可用: {WANDB_AVAILABLE}")
print(f"Python 版本: {sys.version.split()[0]}")

## 1. Experiment 数据类

**核心概念**: `Experiment` 是存储实验信息的数据结构

```
Experiment 数据结构:
┌─────────────────────────────────────────────────────────────┐
│  name: str              ← 实验名称（用于分组）              │
│  experiment_id: str     ← 实验唯一ID（自动生成）            │
│  run_id: str            ← 运行唯一ID（自动生成）            │
│  status: ExperimentStatus ← 状态：RUNNING/COMPLETED/FAILED │
│  params: Dict           ← 超参数（训练前设置）              │
│  metrics: Dict[str, List] ← 指标历史（训练中记录）          │
│  artifacts: List[str]   ← 产物文件路径                      │
│  tags: Dict             ← 标签（用于分类和搜索）            │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 创建 Experiment 实例
# ============================================================
# Experiment 是一个 dataclass，用于存储实验的所有信息
# 创建时只需要指定名称，其他字段会自动初始化

exp = Experiment(
    name="image_classification",      # 实验名称：用于分组相关实验
    description="ResNet50 图像分类实验",  # 描述：说明实验目的
    tags={"model": "resnet50", "dataset": "cifar10"}  # 标签：便于搜索
)

# 查看实验信息
print("=" * 50)
print("实验信息")
print("=" * 50)
print(f"名称: {exp.name}")
print(f"实验ID: {exp.experiment_id}")  # 自动生成的唯一ID
print(f"运行ID: {exp.run_id}")          # 自动生成的唯一ID
print(f"状态: {exp.status.value}")      # 默认是 RUNNING
print(f"描述: {exp.description}")
print(f"标签: {exp.tags}")
print(f"参数: {exp.params}")            # 初始为空字典
print(f"指标: {exp.metrics}")           # 初始为空字典

In [ ]:
# ============================================================
# 实验状态枚举
# ============================================================
# ExperimentStatus 定义了实验的生命周期状态

print("实验状态类型:")
print("-" * 30)
for status in ExperimentStatus:
    print(f"  {status.name:12} → {status.value}")

print("\n状态说明:")
print("  RUNNING   : 实验正在进行中")
print("  COMPLETED : 实验成功完成")
print("  FAILED    : 实验失败（如训练崩溃）")

## 2. ExperimentTracker 本地追踪器

**核心概念**: `ExperimentTracker` 将实验数据保存到本地文件系统

```
ExperimentTracker 工作流程:
┌─────────────────────────────────────────────────────────────┐
│  1. 创建追踪器 → 指定实验名称和保存目录                      │
│  2. log_params() → 记录超参数（训练前调用一次）             │
│  3. log_metrics() → 记录指标（训练中多次调用）              │
│  4. log_artifact() → 记录文件（模型、配置等）               │
│  5. end_run() → 结束并保存实验记录                          │
└─────────────────────────────────────────────────────────────┘

文件结构:
experiments/
└── demo_experiment/
    └── run_001/
        ├── experiment.json  ← 实验元数据
        └── artifacts/       ← 产物文件
```

In [ ]:
# ============================================================
# 创建 ExperimentTracker 实例
# ============================================================

# 创建临时目录用于演示（实际使用时指定固定目录）
demo_dir = tempfile.mkdtemp(prefix="mlops_demo_")
print(f"演示目录: {demo_dir}")

# 创建追踪器
# - experiment_name: 实验名称，用于分组相关实验
# - save_dir: 保存目录
# - run_name: 运行名称，同一实验可以有多次运行
# - description: 描述
# - tags: 标签，便于搜索和分类
tracker = ExperimentTracker(
    experiment_name="demo_experiment",
    save_dir=demo_dir,
    run_name="run_001",
    description="演示实验：展示实验追踪的基本功能",
    tags={"env": "demo", "purpose": "tutorial"}
)

print("\n" + "=" * 50)
print("追踪器创建成功!")
print("=" * 50)
print(f"实验名称: {tracker.experiment_name}")
print(f"运行名称: {tracker.run_name}")
print(f"运行目录: {tracker.run_dir}")

### 2.1 记录参数 (log_params)

**关键区分**:
- **参数 (Parameters)**: 训练**前**设置的配置，不会变化
- **指标 (Metrics)**: 训练**中**产生的数值，随时间变化

```python
# 参数示例
params = {
    "learning_rate": 0.001,  # 学习率
    "batch_size": 32,        # 批次大小
    "epochs": 100,           # 训练轮数
    "optimizer": "adam",     # 优化器
}
```

In [ ]:
# ============================================================
# 记录超参数
# ============================================================
# log_params() 接受一个字典，记录所有超参数
# 可以多次调用，后面的会覆盖前面的同名参数

tracker.log_params({
    "model": "resnet50",       # 模型架构
    "learning_rate": 0.001,    # 学习率
    "batch_size": 32,          # 批次大小
    "epochs": 100,             # 训练轮数
    "optimizer": "adam",       # 优化器
    "weight_decay": 1e-4,      # 权重衰减（L2正则化）
})

# 也可以单独记录单个参数
tracker.log_param("dropout", 0.5)

# 查看记录的参数
print("=" * 50)
print("记录的超参数")
print("=" * 50)
for key, value in tracker.get_experiment().params.items():
    print(f"  {key}: {value}")

### 2.2 记录指标 (log_metrics)

**核心概念**: 指标是训练过程中产生的数值，需要多次记录以追踪变化趋势

```
常见指标:
┌─────────────────────────────────────────────────────────────┐
│  训练指标              验证指标              其他指标        │
│  ──────────           ──────────            ──────────      │
│  train_loss           val_loss              learning_rate   │
│  train_accuracy       val_accuracy          epoch           │
│  train_f1             val_f1                step            │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 模拟训练过程并记录指标
# ============================================================
import random

print("=" * 50)
print("模拟训练过程")
print("=" * 50)
print(f"{'Epoch':<8} {'Train Loss':<12} {'Val Loss':<12} {'Accuracy':<10}")
print("-" * 50)

for epoch in range(10):
    # 模拟指标（实际训练中这些值来自模型评估）
    # 损失随训练逐渐下降，准确率逐渐上升
    train_loss = 1.0 / (epoch + 1) + random.uniform(-0.1, 0.1)
    val_loss = 1.2 / (epoch + 1) + random.uniform(-0.1, 0.1)
    accuracy = 0.5 + epoch * 0.05 + random.uniform(-0.02, 0.02)
    
    # 记录指标
    # - 第一个参数是字典，包含要记录的指标
    # - step 参数指定当前步数（用于绘图时的x轴）
    tracker.log_metrics({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": accuracy,
    }, step=epoch)
    
    print(f"{epoch:<8} {train_loss:<12.4f} {val_loss:<12.4f} {accuracy:<10.4f}")

print("-" * 50)
print("训练完成!")

In [ ]:
# ============================================================
# 查看记录的指标
# ============================================================
# metrics 是一个字典，key 是指标名称，value 是该指标的历史值列表

exp = tracker.get_experiment()

print("=" * 50)
print("记录的指标统计")
print("=" * 50)
for metric_name, values in exp.metrics.items():
    print(f"{metric_name}:")
    print(f"  数据点数量: {len(values)}")
    print(f"  最小值: {min(values):.4f}")
    print(f"  最大值: {max(values):.4f}")
    print(f"  最终值: {values[-1]:.4f}")

### 2.3 获取最佳指标

**实用功能**: 自动找出训练过程中的最佳指标值及其对应的步数

- `mode="min"`: 找最小值（适用于 loss）
- `mode="max"`: 找最大值（适用于 accuracy）

In [ ]:
# ============================================================
# 获取最佳指标
# ============================================================
# get_best_metric() 自动找出训练过程中的最佳值
# - mode="min": 找最小值（适用于 loss）
# - mode="max": 找最大值（适用于 accuracy）

best_loss = tracker.get_best_metric("val_loss", mode="min")
best_acc = tracker.get_best_metric("accuracy", mode="max")

print("=" * 50)
print("最佳指标")
print("=" * 50)
print(f"最低验证损失: {best_loss['value']:.4f} (在 epoch {best_loss['step']})")
print(f"最高准确率:   {best_acc['value']:.4f} (在 epoch {best_acc['step']})")
print("\n提示: 通常在最佳验证损失对应的 epoch 保存模型检查点")

### 2.4 记录文件 (log_artifact)

**用途**: 记录训练过程中产生的文件，如：
- 模型文件 (model.pt, checkpoint.pth)
- 配置文件 (config.yaml)
- 可视化图表 (loss_curve.png)
- 预测结果 (predictions.csv)

In [ ]:
# ============================================================
# 记录产物文件
# ============================================================
# log_artifact() 记录文件路径，便于后续查找

# 创建测试配置文件
config_file = Path(demo_dir) / "config.yaml"
config_file.write_text("""# 训练配置
model: resnet50
learning_rate: 0.001
batch_size: 32
epochs: 100
optimizer: adam
""")

# 记录文件
tracker.log_artifact(str(config_file))

# 查看记录的文件
print("=" * 50)
print("记录的产物文件")
print("=" * 50)
for artifact in tracker.get_experiment().artifacts:
    print(f"  {artifact}")

### 2.5 设置标签 (set_tags)

**用途**: 为实验添加标签，便于分类和搜索

常用标签示例：
- `env`: 环境 (dev, staging, production)
- `status`: 状态 (baseline, experiment, best)
- `team`: 团队 (cv, nlp, rec)
- `priority`: 优先级 (high, medium, low)

In [ ]:
# ============================================================
# 设置标签
# ============================================================
# set_tags() 为实验添加标签，便于后续搜索和分类

tracker.set_tags({
    "status": "baseline",    # 标记为基线实验
    "priority": "high",      # 高优先级
    "team": "cv"             # CV团队
})

# 查看所有标签
print("=" * 50)
print("实验标签")
print("=" * 50)
for key, value in tracker.get_experiment().tags.items():
    print(f"  {key}: {value}")

### 2.6 结束运行 (end_run)

**重要**: 实验结束时必须调用 `end_run()` 来：
1. 设置实验状态（COMPLETED 或 FAILED）
2. 记录结束时间
3. 保存实验数据到文件

In [ ]:
# ============================================================
# 结束运行
# ============================================================
# end_run() 结束实验并保存数据
# 参数是实验状态：COMPLETED（成功）或 FAILED（失败）

tracker.end_run(ExperimentStatus.COMPLETED)

exp = tracker.get_experiment()
print("=" * 50)
print("实验已结束")
print("=" * 50)
print(f"状态: {exp.status.value}")
print(f"持续时间: {exp.duration:.2f} 秒")
print(f"保存位置: {tracker.run_dir}")

## 3. 实验比较 (ExperimentComparator)

**核心功能**: 对比多个实验的参数和指标，找出最佳配置

```
实验比较流程:
┌─────────────────────────────────────────────────────────────┐
│  1. 添加多个实验 → add_experiment(name, params, metrics)    │
│  2. 按指标排序 → compare(metric_name, ascending)            │
│  3. 找最佳实验 → get_best(metric_name, mode)                │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 创建多个实验用于比较
# ============================================================
# 模拟超参数搜索：不同学习率和批次大小的组合

experiments_data = [
    {"name": "exp_lr_0.001",   "params": {"lr": 0.001,  "batch_size": 32}, "metrics": {"accuracy": 0.92, "loss": 0.25}},
    {"name": "exp_lr_0.01",    "params": {"lr": 0.01,   "batch_size": 32}, "metrics": {"accuracy": 0.89, "loss": 0.35}},
    {"name": "exp_lr_0.0001",  "params": {"lr": 0.0001, "batch_size": 32}, "metrics": {"accuracy": 0.95, "loss": 0.18}},
    {"name": "exp_batch_64",   "params": {"lr": 0.001,  "batch_size": 64}, "metrics": {"accuracy": 0.93, "loss": 0.22}},
]

# 创建比较器并添加实验
comparator = ExperimentComparator()

for exp_data in experiments_data:
    comparator.add_experiment(
        exp_data["name"],
        exp_data["params"],
        exp_data["metrics"]
    )

print(f"已添加 {len(experiments_data)} 个实验到比较器")

In [ ]:
# ============================================================
# 比较实验 - 按准确率排序
# ============================================================
# compare() 返回按指定指标排序的实验列表
# - ascending=False: 降序（准确率越高越好）
# - ascending=True: 升序（损失越低越好）

results = comparator.compare(metric_name="accuracy", ascending=False)

print("=" * 70)
print("实验比较 (按准确率降序)")
print("=" * 70)
print(f"{'实验名称':<20} {'学习率':<12} {'批次大小':<10} {'准确率':<10} {'损失':<10}")
print("-" * 70)
for r in results:
    print(f"{r['name']:<20} {r['lr']:<12} {r['batch_size']:<10} {r['accuracy']:<10.4f} {r['loss']:<10.4f}")
print("-" * 70)

In [ ]:
# ============================================================
# 获取最佳实验
# ============================================================
# get_best() 返回指定指标最优的实验
# - mode="max": 找最大值（准确率）
# - mode="min": 找最小值（损失）

best = comparator.get_best("accuracy", mode="max")

print("=" * 50)
print("最佳实验")
print("=" * 50)
print(f"实验名称: {best['name']}")
print(f"准确率:   {best['accuracy']:.4f}")
print(f"损失:     {best['loss']:.4f}")
print(f"学习率:   {best['lr']}")
print(f"批次大小: {best['batch_size']}")
print("\n结论: 学习率 0.0001 在本次实验中表现最佳")

## 4. 加载历史实验

**实用功能**: 从文件系统加载之前保存的实验记录

- `load_experiment(path)`: 加载单个实验
- `list_experiments(dir)`: 列出目录下所有实验

In [ ]:
# ============================================================
# 加载之前保存的实验
# ============================================================
# load_experiment() 从文件系统加载实验记录

loaded_exp = ExperimentTracker.load_experiment(str(tracker.run_dir))

print("=" * 50)
print("加载的实验信息")
print("=" * 50)
print(f"名称: {loaded_exp.name}")
print(f"状态: {loaded_exp.status.value}")
print(f"参数数量: {len(loaded_exp.params)}")
print(f"指标数量: {len(loaded_exp.metrics)}")
print(f"产物数量: {len(loaded_exp.artifacts)}")

In [ ]:
# ============================================================
# 列出所有实验
# ============================================================
# list_experiments() 列出目录下所有实验

all_experiments = ExperimentTracker.list_experiments(demo_dir)

print("=" * 50)
print(f"找到 {len(all_experiments)} 个实验")
print("=" * 50)
for exp_info in all_experiments:
    print(f"  {exp_info['name']}/{exp_info['run_name']}: {exp_info['status']}")

## 5. 工厂函数 (create_tracker)

**便捷功能**: 使用工厂函数快速创建追踪器

```python
# 支持的后端
create_tracker(name, backend="local")   # 本地文件系统
create_tracker(name, backend="mlflow")  # MLflow（需安装）
create_tracker(name, backend="wandb")   # W&B（需安装）
```

In [ ]:
# ============================================================
# 使用工厂函数创建追踪器
# ============================================================
# create_tracker() 是创建追踪器的便捷方法

tracker2 = create_tracker(
    "factory_experiment",
    backend="local",      # 使用本地文件系统
    save_dir=demo_dir
)

# 快速记录一些数据
tracker2.log_params({"test_param": "value"})
tracker2.log_metric("test_score", 0.95)
tracker2.end_run()

print("工厂函数创建的追踪器运行成功!")

## 6. MLflow 集成 (可选)

In [ ]:
if MLFLOW_AVAILABLE:
    from experiment_tracker import MLflowTracker
    
    print("MLflow 追踪器使用示例:")
    print("""
# 创建 MLflow 追踪器
tracker = MLflowTracker(
    experiment_name="my_experiment",
    tracking_uri="http://localhost:5000",  # MLflow 服务器地址
    run_name="run_001"
)

# 记录参数和指标
tracker.log_params({"lr": 0.001})
tracker.log_metrics({"loss": 0.5}, step=0)

# 记录模型
tracker.log_model(model, "model")

# 结束运行
tracker.end_run()
""")
else:
    print("MLflow 未安装")
    print("安装命令: pip install mlflow")

## 7. 清理演示目录

In [ ]:
# 清理
shutil.rmtree(demo_dir, ignore_errors=True)
print("演示目录已清理")

## 总结

本教程介绍了实验追踪的核心功能：

1. **Experiment 数据类**: 存储实验信息
2. **ExperimentTracker**: 本地文件系统追踪
3. **参数和指标记录**: log_params, log_metrics
4. **文件记录**: log_artifact
5. **实验比较**: ExperimentComparator
6. **MLflow 集成**: 企业级追踪

### 最佳实践

- 在训练开始时记录所有超参数
- 定期记录训练指标
- 保存最佳模型检查点
- 使用有意义的实验名称和标签
- 记录数据版本和代码版本